# CHGNet-PyG Foundation Potential

This notebook demonstrates CHGNet with the PyTorch Geometric (PyG) backend. CHGNet is a
charge-informed graph neural network potential that jointly predicts energy, forces, stresses,
and site-wise **magnetic moments**.

We cover:
1. **Static prediction** – energy / forces / stresses / magnetic moments for a single structure
2. **Structure relaxation** – ionic + cell relaxation via the ASE `Relaxer`
3. **Molecular dynamics** – NVT trajectory via the ASE `MolecularDynamics` driver
4. **Training from scratch** – fit a small CHGNet on a custom dataset
5. **Fine-tuning** – continue training from the pre-trained MatPES checkpoint

**Reference:**  
Deng, B. et al. *CHGNet as a pretrained universal neural network potential for charge-informed
atomistic modelling.* Nat. Mach. Intell. (2023) doi:10.1038/s42256-023-00716-3

Author: Bowen Deng

In [ ]:
from __future__ import annotations

import os
import warnings

import numpy as np
import torch
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from pymatgen.core import Lattice, Structure
from pymatgen.io.ase import AseAtomsAdaptor

import matgl
from matgl.ext._ase_pyg import MolecularDynamics, PESCalculator, Relaxer

warnings.filterwarnings("ignore")

/home/bdeng/anaconda3/matgl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/bdeng/anaconda3/matgl/lib/python3.11/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()


## 1. Load the pre-trained CHGNet-PyG model

Two MatPES checkpoints are available:

| Model name | Training functional |
|---|---|
| `BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10` | r2SCAN |
| `BowenD-UCB/CHGNet-PyG-MatPES-PBE-2025.2.10` | PBE |

Weights are numerically identical to the DGL checkpoints; the only difference is the
message-passing backend. All predictions are in **eV** (energy), **eV/Å** (forces),
**GPa** (stresses), and **μB** (magnetic moments).

In [ ]:
pot = matgl.load_model("BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10")
print(pot)

Potential(
  (model): CHGNet(
    (bond_expansion): RadialBesselFunction()
    (threebody_bond_expansion): RadialBesselFunction()
    (angle_expansion): FourierExpansion()
    (atom_embedding): Embedding(89, 128)
    (bond_embedding): _MLPNorm(
      (layers): ModuleList(
        (0): Linear(in_features=63, out_features=128, bias=False)
      )
      (activation): SiLU()
    )
    (angle_embedding): _MLPNorm(
      (layers): ModuleList(
        (0): Linear(in_features=65, out_features=128, bias=False)
      )
      (activation): SiLU()
    )
    (atom_bond_weights): Linear(in_features=63, out_features=128, bias=False)
    (bond_bond_weights): Linear(in_features=63, out_features=128, bias=False)
    (threebody_bond_weights): Linear(in_features=63, out_features=128, bias=False)
    (atom_graph_layers): ModuleList(
      (0-4): 5 x CHGNetAtomGraphBlock(
        (conv): CHGNetGraphConv(
          (node_update_func): _GatedMLPNorm(
            (value): _MLPNorm(
              (layers): Modu

## 2. Static prediction

Use `PESCalculator` (an ASE `Calculator` wrapper) to obtain energy, forces,
stresses, and magnetic moments for any pymatgen `Structure`.

In [ ]:
# BCC iron — a classic magnetic test case
fe_struct = Structure(
    Lattice.cubic(2.87),
    ["Fe", "Fe"],
    [[0, 0, 0], [0.5, 0.5, 0.5]],
)

adaptor = AseAtomsAdaptor()
fe_atoms = adaptor.get_atoms(fe_struct)

calc = PESCalculator(potential=pot)
fe_atoms.set_calculator(calc)

energy = fe_atoms.get_potential_energy()  # eV
forces = fe_atoms.get_forces()  # eV/Å
magmoms = fe_atoms.calc.results["magmoms"]  # μB per site

print(f"Energy          : {energy:.4f} eV  ({energy / len(fe_struct):.4f} eV/atom)")
print(f"Max |force|     : {np.abs(forces).max():.4e} eV/Å")
print(f"Magnetic moments: {magmoms.flatten().tolist()}  μB")

The stress unit is now in GPa. Please set stress_unit='eV/A3' if you want to use PESCalculator for other ASE applications.


Energy          : -28.8047 eV  (-14.4023 eV/atom)
Max |force|     : 1.7136e-07 eV/Å
Magnetic moments: [2.7359468936920166, 2.7359461784362793]  μB


### 2a. Direct Potential call (without ASE)

You can also call the `Potential` directly with a PyG graph for finer control
(e.g. batching, or when you need raw tensors for downstream differentiable ops).

In [ ]:
from matgl.ext._pymatgen_pyg import Structure2Graph

conv = Structure2Graph(
    element_types=pot.model.element_types,
    cutoff=pot.model.cutoff,
)

g, lat, state = conv.get_graph(fe_struct)
# Attach Cartesian positions and PBC shift vectors required by Potential
g.pbc_offshift = torch.matmul(g.pbc_offset, lat[0])
g.pos = g.frac_coords @ lat[0]

# out = (energy, forces, stresses, hessian[unused], magmom)
out = pot(g=g, lat=lat, state_attr=state)
energy_t, forces_t, stress_t, _, magmom_t = out

print(f"Energy/atom : {energy_t.item() / len(fe_struct):.4f} eV")
print(f"Forces (eV/Å):\n{forces_t.detach().numpy()}")
print(f"Stress (GPa):\n{stress_t.detach().numpy()}")
print(f"Magmom (μB) : {magmom_t.detach().flatten().tolist()}")

Energy/atom : -14.4023 eV
Forces (eV/Å):
[[-2.9802322e-08 -3.7252903e-08 -1.4901161e-08]
 [-0.0000000e+00  7.4505806e-09  7.4505806e-09]]
Stress (GPa):
[[ 1.2303020e+00 -7.2461411e-08  3.6230705e-07]
 [-7.2461410e-07  1.2303020e+00 -2.1738423e-07]
 [ 0.0000000e+00 -7.2461411e-08  1.2303011e+00]]
Magmom (μB) : [2.7359468936920166, 2.7359461784362793]


## 3. Structure relaxation

`Relaxer` wraps the ASE cell-filter + FIRE optimizer. By default it relaxes both
ionic positions *and* the cell (`relax_cell=True`).

In [ ]:
# Deliberately distorted CsCl structure
struct = Structure(
    Lattice.cubic(4.5),
    ["Cs", "Cl"],
    [[0, 0, 0], [0.48, 0.52, 0.50]],  # slightly off-centre
)

relaxer = Relaxer(potential=pot)
result = relaxer.relax(struct, fmax=0.01, steps=500)

final_struct = result["final_structure"]
traj = result["trajectory"]

print(f"Initial energy  : {traj.energies[0]:.4f} eV")
print(f"Final energy    : {traj.energies[-1]:.4f} eV")
print(f"Relaxation steps: {len(traj.energies)}")
print(f"Final structure :\n{final_struct}")

Initial energy  : -35.3086 eV
Final energy    : -35.4979 eV
Relaxation steps: 44
Final structure :
Full Formula (Cs1 Cl1)
Reduced Formula: CsCl
abc   :   4.126269   4.126269   4.126277
angles:  90.000000  90.000000  90.107797
pbc   :       True       True       True
Sites (2)
  #  SP            a         b     c  final_magmom
---  ----  ---------  --------  ----  --------------
  0  Cs    -0.010176  0.010176  -0    [-0.00871292]
  1  Cl     0.490176  0.509824   0.5  [-0.01081806]


In [ ]:
# Trajectory data as a DataFrame
df = traj.as_pandas()
df[["energies", "forces"]].head()

,energies,forces
0,-35.308643,"[[0.0070515014, -0.0070515377, -0.0], [-0.0070..."
1,-35.322723,"[[0.0032790117, -0.0032787677, -0.0], [-0.0032..."
2,-35.348724,"[[-0.00494536, 0.0049444847, -0.0], [0.0049453..."
3,-35.382706,"[[-0.014665338, 0.0146656595, -0.0], [0.014665..."
4,-35.420620,"[[-0.025606597, 0.025606595, -0.0], [0.0256065..."


## 4. Molecular dynamics

Run a short NVT simulation (Nose-Hoover thermostat) on the relaxed structure.
Ensembles available: `"nve"`, `"nvt"`, `"nvt_langevin"`, `"npt"`, `"npt_nose_hoover"`, …

In [ ]:
# Convert the relaxed pymatgen structure to ASE Atoms
atoms = adaptor.get_atoms(final_struct)

# Initialise velocities from a Maxwell-Boltzmann distribution at 300 K
MaxwellBoltzmannDistribution(atoms, temperature_K=300)

driver = MolecularDynamics(
    atoms=atoms,
    potential=pot,
    ensemble="nvt",  # Nose-Hoover NVT
    temperature=300,  # K
    timestep=2.0,  # fs
    taut=100,  # thermostat time constant (fs)
    logfile="md.log",
    loginterval=10,
)

driver.run(steps=100)
print("MD finished. Final potential energy:", atoms.get_potential_energy(), "eV")

MD finished. Final potential energy: -35.33189010620117 eV


## 5. Training a CHGNet model from scratch

We demonstrate training on a small synthetic dataset. In a real workflow, replace
`structures` / `energies` / `forces` / `stresses` with your own DFT data (or download
from the Materials Project via `mp_api`).

Set `magmom_weight > 0` and include `"magmoms"` in the label dict to train the
magnetic moment head.

In [ ]:
from matgl.ext.pymatgen import get_element_list
from matgl.graph.data import MGLDataset
from matgl.models._chgnet_pyg import CHGNet
from matgl.utils.training import MGLPotentialTrainer, fit_element_refs

In [ ]:
# Small synthetic dataset for demonstration
# Replace with real DFT data or an mp_api download in production
train_structures = [
    Structure(Lattice.cubic(4.0), ["Mo", "S"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]),
    Structure(Lattice.cubic(4.0), ["Mo", "S"], [[0.01, 0.0, 0.0], [0.51, 0.5, 0.5]]),
    Structure(Lattice.cubic(4.0), ["Mo", "S"], [[0.0, 0.02, 0.0], [0.5, 0.52, 0.5]]),
    Structure(Lattice.cubic(4.0), ["Mo", "S"], [[0.0, 0.0, -0.01], [0.5, 0.5, 0.49]]),
    Structure(Lattice.cubic(3.95), ["Mo", "S"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]),
    Structure(Lattice.cubic(4.05), ["Mo", "S"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]),
]
train_energies = [-10.00, -9.98, -9.97, -9.99, -9.95, -9.96]
train_forces = [np.zeros((2, 3)).tolist() for _ in train_structures]
train_stresses = [np.zeros((3, 3)).tolist() for _ in train_structures]

element_types = get_element_list(train_structures)
print(f"Element types: {element_types}")

Element types: ('S', 'Mo')


In [ ]:
# Fit per-element energy offsets so the model learns residuals only
atomrefs = fit_element_refs(train_structures, train_energies, element_types)
print(f"Element refs (eV): {dict(zip(element_types, atomrefs.tolist(), strict=True))}")

# Build PyG graphs with three-body (line-graph) support
converter = Structure2Graph(element_types=element_types, cutoff=6.0)
dataset = MGLDataset(
    structures=train_structures,
    converter=converter,
    labels={"energies": train_energies, "forces": train_forces, "stresses": train_stresses},
    include_line_graph=True,
    save_cache=False,
)
print(f"Dataset: {len(dataset)} structures")

Processing...


Element refs (eV): {'S': -4.987499999999998, 'Mo': -4.9875}


  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 2776.15it/s]

Dataset: 6 structures



Done!


In [ ]:
# Small CHGNet for demonstration (2 blocks)
model = CHGNet(
    element_types=element_types,
    num_blocks=2,
    dim_atom_embedding=64,
    dim_bond_embedding=64,
    dim_angle_embedding=64,
)

trainer = MGLPotentialTrainer(
    model=model,
    energy_weight=1.0,
    force_weight=1.0,
    stress_weight=0.1,
    magmom_weight=0.0,  # set > 0 when magmom labels are available
    loss="huber_loss",
    lr=1e-3,
    max_epochs=5,  # small for demo; use 100+ for real training
    accelerator="cpu",  # change to "gpu" / "cuda" on a GPU machine
)

# Lightning collects CUDA RNG states even when accelerator="cpu" if a CUDA
# device is visible but its driver is incompatible. Hide all GPUs to avoid
# the crash; remove this line (or set to your device id) on a GPU machine.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

potential = trainer.fit(
    dataset=dataset,
    atomrefs=atomrefs,
    save_path="./trained_chgnet_pyg/",
)
print("Training complete.")

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ mae   │ MeanAbsoluteError │      0 │ train │     0 │
│ 1 │ rmse  │ MeanSquaredError  │      0 │ train │     0 │
│ 2 │ model │ Potential         │  163 K │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 163 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 163 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 79                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=5` reached.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_Charge_MAE      │            0.0            │
│     test_Charge_RMSE      │            0.0            │
│      test_Energy_MAE      │     0.000213623046875     │
│     test_Energy_RMSE      │     0.000213623046875     │
│      test_Force_MAE       │  3.3954467709707004e-11   │
│      test_Force_RMSE      │  5.4448282688079885e-11   │
│      test_Magmom_MAE      │            0.0            │
│     test_Magmom_RMSE      │            0.0            │
│      test_Stress_MAE      │   0.0021507737692445517   │
│     test_Stress_RMSE      │   0.003725249320268631    │
│      test_Total_Loss      │   7.166915452216926e-07   │
└───────────────────────────┴───────────────────────────┘

Training complete.


In [ ]:
# Load the saved model back
loaded_pot = matgl.load_model("./trained_chgnet_pyg/")
print(loaded_pot)

Potential(
  (model): CHGNet(
    (bond_expansion): RadialBesselFunction()
    (threebody_bond_expansion): RadialBesselFunction()
    (angle_expansion): FourierExpansion()
    (atom_embedding): Embedding(2, 64)
    (bond_embedding): _MLPNorm(
      (layers): ModuleList(
        (0): Linear(in_features=9, out_features=64, bias=False)
      )
      (activation): SiLU()
    )
    (angle_embedding): _MLPNorm(
      (layers): ModuleList(
        (0): Linear(in_features=9, out_features=64, bias=False)
      )
      (activation): SiLU()
    )
    (atom_bond_weights): Linear(in_features=9, out_features=64, bias=False)
    (bond_bond_weights): Linear(in_features=9, out_features=64, bias=False)
    (threebody_bond_weights): Linear(in_features=9, out_features=64, bias=False)
    (atom_graph_layers): ModuleList(
      (0-1): 2 x CHGNetAtomGraphBlock(
        (conv): CHGNetGraphConv(
          (node_update_func): _GatedMLPNorm(
            (value): _MLPNorm(
              (layers): ModuleList(
    

## 6. Fine-tuning the pre-trained CHGNet-PyG

Start from the MatPES r2SCAN checkpoint and continue training on your own dataset.
A lower learning rate (`lr=1e-4`) prevents the pre-trained weights from drifting too fast.

Since the pre-trained CHGNet was trained on the full periodic table, make sure
`element_types` here matches the pre-trained model's `element_types` (or is a subset of it).

In [ ]:
# Load the pre-trained potential
pretrained_pot = matgl.load_model("BowenD-UCB/CHGNet-PyG-MatPES-r2SCAN-2025.2.10")
pretrained_model = pretrained_pot.model

# Per-element energy references from the pre-trained checkpoint
property_offset = pretrained_pot.element_refs.property_offset.numpy()

# Build dataset using the same cutoffs as the pre-trained model
ft_converter = Structure2Graph(
    element_types=pretrained_model.element_types,
    cutoff=pretrained_model.cutoff,
)
ft_dataset = MGLDataset(
    structures=train_structures,
    converter=ft_converter,
    labels={"energies": train_energies, "forces": train_forces, "stresses": train_stresses},
    include_line_graph=True,
    save_cache=False,
)

ft_trainer = MGLPotentialTrainer(
    model=pretrained_model,
    energy_weight=1.0,
    force_weight=1.0,
    stress_weight=0.1,
    magmom_weight=0.0,  # set > 0 and add "magmoms" to labels to train the magmom head
    lr=1e-4,  # lower LR for fine-tuning
    max_epochs=5,
    accelerator="cpu",
)

ft_potential = ft_trainer.fit(
    dataset=ft_dataset,
    atomrefs=property_offset,
    save_path="./finetuned_chgnet_pyg/",
)
print("Fine-tuning complete.")

Processing...


  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 3113.43it/s]


Done!
Seed set to 42


GPU available: False, used: False


...

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ mae   │ MeanAbsoluteError │      0 │ train │     0 │
│ 1 │ rmse  │ MeanSquaredError  │      0 │ train │     0 │
│ 2 │ model │ Potential         │  2.7 M │ train │     0 │
└───┴───────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 2.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.7 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 322                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=5` reached.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_Charge_MAE      │            0.0            │
│     test_Charge_RMSE      │            0.0            │
│      test_Energy_MAE      │    11.532892227172852     │
│     test_Energy_RMSE      │    11.532892227172852     │
│      test_Force_MAE       │  4.6566128730773926e-09   │
│      test_Force_RMSE      │   6.224319726300109e-09   │
│      test_Magmom_MAE      │            0.0            │
│     test_Magmom_RMSE      │            0.0            │
│      test_Stress_MAE      │    0.9226765036582947     │
│     test_Stress_RMSE      │    1.5981225967407227     │
│      test_Total_Loss      │    11.108492851257324     │
└───────────────────────────┴───────────────────────────┘

Fine-tuning complete.


## 7. Cleanup

In [ ]:
import os
import shutil

for path in ("md.log", "trained_chgnet_pyg", "finetuned_chgnet_pyg"):
    if os.path.isfile(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)